# Power Outage Analysis

**Name(s)**: Alina Gao, Fei Liang

**Website Link**: (your website link)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

#from dsc80_utils import openpyxl # Feel free to uncomment and use this.

In [3]:
# pip install openpyxl

## Step 1: Introduction

This project analyzes a dataset of major power outages in the United States from 2000 to 2016. The dataset contains 1534 outage events with 56 variables, including outage duration, cause category, climate region, and the number of customers affected.

Our research question is: Does the cause of a power outage (specifically severe weather versus other causes) significantly affect how long the outage lasts?

Power outages affect millions of people every year. When the power goes out, hospitals lose electricity, food spoils, and people lose heat or air conditioning. Knowing what causes the longest outages can help power companies prepare better and fix problems faster. Since storms and extreme weather are becoming more common, it is especially important to understand whether weather-related outages tend to last longer than outages from other causes.

## Step 2: Data Cleaning and Exploratory Data Analysis

In [4]:
df = pd.read_excel('outage.xlsx', skiprows=5, header=0)
df = df.drop(0).reset_index(drop=True)

print(df.shape)
df.head()

(1534, 57)


,variables,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,...,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
0,NaN,1.0,2011.0,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
1,NaN,2.0,2014.0,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
2,NaN,3.0,2010.0,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
3,NaN,4.0,2012.0,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
4,NaN,5.0,2015.0,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743


In [5]:
df = df.drop(columns='variables')

print(df.shape)
df.head()

(1534, 56)


,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,...,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
0,1.0,2011.0,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
1,2.0,2014.0,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
2,3.0,2010.0,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
3,4.0,2012.0,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2012-06-19 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
4,5.0,2015.0,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,2015-07-18 00:00:00,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743


In [6]:
fig = px.histogram(
    df,
    x='CAUSE.CATEGORY',
    title='Number of Power Outages by Cause Category',
    labels={'CAUSE.CATEGORY': 'Cause Category', 'count': 'Count'},
    color='CAUSE.CATEGORY'
)
fig.update_layout(showlegend=False)
fig.show()

## Step 3: Assessment of Missingness

**NMAR (Not Missing At Random) Analysis**<br>
Maybe some utility company negligence or liability-inducing equipment failure, the company may intentionally leave the detailed cause blank.
We believe `DEMAND.LOSS.MW` is likely **NMAR**. Utilities may intentionally omit demand 
loss figures when outages result from operational errors, to limit liability exposure. 
The value is more likely missing *because* the loss was large or caused by negligence — 
driven by the unobserved value itself. Additional data such as regulatory incident reports 
could help explain this missingness, making it MAR.
 

In [7]:
df['demand_missing'] = df['DEMAND.LOSS.MW'].isna()

#Test 1: CAUSE.CATEGORY (TVD，expect significant
def tvd(a, b):
    cats = set(a.index) | set(b.index)
    return sum(abs(a.get(c,0) - b.get(c,0)) for c in cats) / 2

tmp = df[['demand_missing','CAUSE.CATEGORY']].dropna().reset_index(drop=True)
obs_tvd = tvd(tmp[tmp.demand_missing]['CAUSE.CATEGORY'].value_counts(normalize=True),
              tmp[~tmp.demand_missing]['CAUSE.CATEGORY'].value_counts(normalize=True))

rng = np.random.default_rng(42)
sims_tvd = [tvd(
    pd.Series(tmp['CAUSE.CATEGORY'].values[rng.permutation(len(tmp))[:tmp.demand_missing.sum()]]).value_counts(normalize=True),
    pd.Series(tmp['CAUSE.CATEGORY'].values[rng.permutation(len(tmp))[tmp.demand_missing.sum():]]).value_counts(normalize=True)
) for _ in range(1000)]

p1 = np.mean(np.array(sims_tvd) >= obs_tvd)
print(f"Test 1 | CAUSE.CATEGORY | TVD={obs_tvd:.3f} | p={p1:.3f}")

#Test 2: OUTAGE.DURATION (abs diff in means，expect NOT significant)
tmp2 = df[['demand_missing','OUTAGE.DURATION']].dropna().reset_index(drop=True)
vals = tmp2['OUTAGE.DURATION'].values.astype(float)
mask = tmp2['demand_missing'].values.astype(bool)
obs_diff = abs(vals[mask].mean() - vals[~mask].mean())

sims_diff = [abs(vals[s].mean() - vals[~s].mean())
             for s in (rng.permutation(mask) for _ in range(1000))]  

p2 = np.mean(np.array(sims_diff) >= obs_diff)
print(f"Test 2 | OUTAGE.DURATION | diff={obs_diff:.1f} | p={p2:.3f}")

Test 1 | CAUSE.CATEGORY | TVD=0.179 | p=0.000
Test 2 | OUTAGE.DURATION | diff=128.8 | p=0.660


In [8]:
fig1 = px.histogram(x=sims_tvd, nbins=40,
    title='Missingness of DEMAND.LOSS.MW vs. CAUSE.CATEGORY',
    labels={'x':'TVD'})
fig1.add_vline(x=obs_tvd, line_dash='dash', line_color='red',
    annotation_text=f'Observed={obs_tvd:.3f}, p≈{p1:.3f}')
fig1.show()

fig2 = px.histogram(x=sims_diff, nbins=40,
    title='Missingness of DEMAND.LOSS.MW vs. OUTAGE.DURATION',
    labels={'x':'|Diff in Means| (minutes)'})
fig2.add_vline(x=obs_diff, line_dash='dash', line_color='red',
    annotation_text=f'Observed={obs_diff:.1f}, p={p2:.3f}')
fig2.show()

## Step 4: Hypothesis Testing

Null Hypothesis : The distribution of outage duration for severe weather events is the same as for non-severe-weather.

Alternative Hypothesis: Power outages caused by severe weather tend to last longer than those caused by other factors.

Test Statistic: Difference in means of outage duration based on weather condition.


In [13]:
df_ht = df[['CAUSE.CATEGORY', 'OUTAGE.DURATION']].dropna()
df_ht['is_weather'] = df_ht['CAUSE.CATEGORY'] == 'severe weather'

observed_diff = (
    df_ht.groupby('is_weather')['OUTAGE.DURATION'].mean()[True]
    - df_ht.groupby('is_weather')['OUTAGE.DURATION'].mean()[False]
)
print(f"Observed difference in means: {observed_diff:.2f} minutes")

n_repetitions = 1000
simulated_diffs = []

for _ in range(n_repetitions):
    shuffled = df_ht['is_weather'].sample(frac=1).reset_index(drop=True)
    temp = df_ht.copy()
    temp['is_weather'] = shuffled
    diff = (
        temp.groupby('is_weather')['OUTAGE.DURATION'].mean()[True]
        - temp.groupby('is_weather')['OUTAGE.DURATION'].mean()[False]
    )
    simulated_diffs.append(diff)

p_value = np.mean(np.array(simulated_diffs) >= observed_diff)
print(f"P-value: {p_value}")
# the p-vlaue is 0.0 so we reject the null hypothesis


Observed difference in means: 2537.81 minutes
P-value: 0.0


## Step 5: Framing a Prediction Problem

We plan to predict the column CAUSE.CATEGORY using features such as climate region, state, month, anomaly level, and number of customers affected. This is a classification problem since CAUSE.CATEGORY contains discrete categories.

## Step 6: Baseline Model

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

In [11]:
df_model= df[['CLIMATE.REGION', 'MONTH', 'CAUSE.CATEGORY']].dropna()

le= LabelEncoder()
df_model= df_model.copy()
df_model['CLIMATE.REGION']= le.fit_transform(df_model['CLIMATE.REGION'])

X= df_model[['CLIMATE.REGION', 'MONTH']]
y= df_model['CAUSE.CATEGORY']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline= DecisionTreeClassifier(max_depth=2, random_state=42)
baseline.fit(X_train, y_train)

y_pred= baseline.predict(X_test)
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Baseline Accuracy: 0.4375


Our baseline model is a Decision Tree classifer with a maximum depth of 2, trained to predict the cause category of a power outage using only two features (climate region and month). We kept this baseline model simple intentionally as a starting point so we have a reference to compare against later when we make improvements. The baseline accuracy is like a benchmark, meaning our final model should perform better than this to be considered worthwhile. We plan to add more features such as state, anomaly level, number of customers affected, and outage duration. We will also increase the tree depth or switch to a better, more powerful algorithm.

## Step 7: Final Model

We add three new features on top of the baseline:
- `ANOMALY.LEVEL`: climate anomaly severity — directly predictive of weather-caused outages
- `OUTAGE.DURATION`: longer outages are more likely caused by severe weather than equipment failure
- `CUSTOMERS.AFFECTED`: large-scale outages are more characteristic of weather events

We switch to RandomForestClassifier and tune `max_depth` and `n_estimators` via GridSearchCV.

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer

features = ['CLIMATE.REGION', 'ANOMALY.LEVEL', 'OUTAGE.DURATION', 'CUSTOMERS.AFFECTED']
target = 'CAUSE.CATEGORY'

df_final = df[['CLIMATE.REGION', 'MONTH', 'CAUSE.CATEGORY',
               'ANOMALY.LEVEL', 'OUTAGE.DURATION', 'CUSTOMERS.AFFECTED']].dropna(
               subset=['CLIMATE.REGION', 'MONTH', 'CAUSE.CATEGORY'])

X = df_final[features]
y = df_final[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline
preprocessor = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore'), ['CLIMATE.REGION']),
    ('num', SimpleImputer(strategy='median'), ['ANOMALY.LEVEL', 'OUTAGE.DURATION', 'CUSTOMERS.AFFECTED'])
])

pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [5, 10, None]
}
gs = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print(f"Baseline Accuracy : 0.4375")
print(f"Final Accuracy    : {gs.score(X_test, y_test):.4f}")

Best params: {'clf__max_depth': None, 'clf__n_estimators': 100}
Baseline Accuracy : 0.4375
Final Accuracy    : 0.7336


## Step 8: Fairness Analysis

In [ ]:
# TODO

: 